**TabNet Classifier**

**Tabnet Model Test data Accuracy after Training on Chunks data**

In [ ]:
import pandas as pd
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# Function to load the model and test it on new data
def test_saved_model(test_file, target_column, model_file):
    # Load the test data
    test_df = pd.read_csv(test_file)

    # Separate features and target from test data
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    # Encode categorical variables if necessary (for test data)
    for column in X_test.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X_test[column] = le.fit_transform(X_test[column])

    # Load the saved TabNet model
    tabnet_model = TabNetClassifier()
    tabnet_model.load_model(model_file)

    # Make predictions on the test data
    test_preds = tabnet_model.predict(X_test.values)

    # Calculate and display test accuracy
    test_accuracy = accuracy_score(y_test.values, test_preds)
    print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

    return test_accuracy

# Define file paths and target column
test_file = 'test_data.csv'  # Path to your test data
target_column = 'vt_detection'  # Replace with the name of your target column
model_file = 'tabnet_model.zip'  # Path to your saved TabNet model

# Test the saved model on the test set
test_accuracy = test_saved_model(test_file, target_column, model_file)


/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/abstract_model.py:454: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_state_dict = torch.load(f, map_location=self.device)
Test Accuracy: 90.39%

**Tabnet Full Dataset Training**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
from sklearn.metrics import accuracy_score

# Function to train TabNet on training data and evaluate on test data
def train_and_evaluate_tabnet(train_file, valid_file, test_file, target_column, max_epochs=50, patience=10):
    # Read the training and test datasets
    train_df = pd.read_csv(train_file)
    valid_df = pd.read_csv(valid_file)
    test_df = pd.read_csv(test_file)

    # Separate features and target from training data
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    # Separate features and target from validation data
    X_valid = valid_df.drop(columns=[target_column])
    y_valid = valid_df[target_column]

    # Separate features and target from test data
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    # Encode categorical variables if necessary (for both training and test data)
    for column in X_train.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        # Fit encoder only on training data
        X_train[column] = le.fit_transform(X_train[column])

        # Use the same encoder to transform valid data (ignore unseen labels)
        X_valid[column] = X_valid[column].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

        # Use the same encoder to transform test data (ignore unseen labels)
        X_test[column] = X_test[column].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
        # This will assign a value of -1 to any unseen label in the test data

    # Initialize the TabNet model
    tabnet_model = TabNetClassifier()

    # Train the model on the full training data
    tabnet_model.fit(
        X_train.values, y_train.values,
        eval_set=[(X_train.values, y_train.values)],
        eval_name=['train'],
        eval_metric=['accuracy'],
        max_epochs=max_epochs,
        patience=patience,
        batch_size=512,  # Adjust batch size to fit in memory
        virtual_batch_size=64  # Helps in managing memory usage
    )

    # Make predictions on the valid data
    valid_preds = tabnet_model.predict(X_valid.values)
    valid_accuracy = accuracy_score(y_valid.values, valid_preds)
    print(f"valid Accuracy: {valid_accuracy * 100:.2f}%")

    #
    # Make predictions on the test data
    test_preds = tabnet_model.predict(X_test.values)

    # Calculate accuracy on the test data
    accuracy = accuracy_score(y_test.values, test_preds)
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

    # Show feature importance
    print("Feature Importance: ", tabnet_model.feature_importances_)

    return tabnet_model

# Define file paths and target column
train_file = 'train_data.csv'  # Path to your train data
valid_file = 'valid_data.csv'  # Path to your validation data
test_file = 'test_data.csv'    # Path to your test data
target_column = 'vt_detection'  # Replace with the name of your target column

# Train the model and evaluate on the test set
tabnet_model = train_and_evaluate_tabnet(train_file, valid_file, test_file, target_column)

# You can save the trained model if needed
tabnet_model.save_model('tabnet_model')


/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.17444 | train_accuracy: 0.90386 |  0:05:52s
epoch 1  | loss: 0.06668 | train_accuracy: 0.9041  |  0:11:36s
epoch 2  | loss: 0.04448 | train_accuracy: 0.99162 |  0:17:19s
epoch 3  | loss: 0.02472 | train_accuracy: 0.99776 |  0:23:05s
epoch 4  | loss: 0.0113  | train_accuracy: 0.99773 |  0:28:47s
epoch 5  | loss: 0.01108 | train_accuracy: 0.99815 |  0:34:33s
epoch 6  | loss: 0.00573 | train_accuracy: 0.99844 |  0:40:15s
epoch 7  | loss: 0.00628 | train_accuracy: 0.99851 |  0:46:01s
epoch 8  | loss: 0.00615 | train_accuracy: 0.99825 |  0:51:41s
epoch 9  | loss: 0.00618 | train_accuracy: 0.99837 |  0:57:29s
epoch 10 | loss: 0.00801 | train_accuracy: 0.998   |  1:03:12s
epoch 11 | loss: 0.00687 | train_accuracy: 0.99819 |  1:08:56s
epoch 12 | loss: 0.00423 | train_accuracy: 0.99854 |  1:14:37s
epoch 13 | loss: 0.00326 | train_accuracy: 0.99819 |  1:20:22s
epoch 14 | loss: 0.00193 | train_accuracy: 0.99905 |  1:26:07s
epoch 15 | loss: 0.00154 | train_accuracy: 0.99922 |  1

/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


**Count Of Malware and Benin in test_data Target Column**

In [ ]:
df['vt_detection'].value_counts()

vt_detection
0    27641
1     2940
Name: count, dtype: int64

**Tabnet Chunk Training**

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

# Function to train TabNet model in chunks
def train_tabnet_in_chunks(file_path, target_column, chunk_size=20000, max_epochs=50, patience=10):
    # Initialize the TabNet model
    tabnet_model = TabNetClassifier()

    # Read the CSV file in chunks
    for chunk_idx, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):
        # Define features and target variable for the current chunk
        X_chunk = chunk.drop(columns=[target_column])
        y_chunk = chunk[target_column]

        # Encode categorical variables if necessary
        for column in X_chunk.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            X_chunk[column] = le.fit_transform(X_chunk[column])

        # Convert data to torch tensors
        X_chunk_tensor = torch.FloatTensor(X_chunk.values)
        y_chunk_tensor = torch.LongTensor(y_chunk.values)

        # Move tensors to GPU if available
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        X_chunk_tensor = X_chunk_tensor.to(device)
        y_chunk_tensor = y_chunk_tensor.to(device)

        # Move tensors back to CPU and convert to NumPy arrays for TabNet
        X_chunk_numpy = X_chunk_tensor.cpu().numpy()
        y_chunk_numpy = y_chunk_tensor.cpu().numpy()

        # Create eval_set and eval_name for the current chunk
        eval_set = [(X_chunk_numpy, y_chunk_numpy)]
        eval_name = [f'train_chunk_{chunk_idx}']

        # Fit the model on the current chunk (in CPU NumPy format)
        tabnet_model.fit(X_chunk_numpy, y_chunk_numpy,
                         eval_set=eval_set,  # Use the current chunk for evaluation
                         eval_name=eval_name,
                         eval_metric=['accuracy'],
                         max_epochs=max_epochs,
                         patience=patience)

    return tabnet_model

# Define file path and target column
file_path = 'train_data.csv'  # Replace with your CSV file path
target_column = 'vt_detection'  # Replace with the name of your target column

# Train the TabNet model in chunks
tabnet_model = train_tabnet_in_chunks(file_path, target_column)

# You can save the trained model if needed
tabnet_model.save_model('tabnet_model')


/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.3432  | train_chunk_0_accuracy: 0.9042  |  0:01:18s
epoch 1  | loss: 0.2051  | train_chunk_0_accuracy: 0.9042  |  0:02:37s
epoch 2  | loss: 0.17722 | train_chunk_0_accuracy: 0.9042  |  0:03:54s
epoch 3  | loss: 0.1473  | train_chunk_0_accuracy: 0.9042  |  0:05:12s
epoch 4  | loss: 0.14227 | train_chunk_0_accuracy: 0.9042  |  0:06:30s
epoch 5  | loss: 0.13391 | train_chunk_0_accuracy: 0.9042  |  0:07:47s
epoch 6  | loss: 0.10988 | train_chunk_0_accuracy: 0.9042  |  0:09:03s
epoch 7  | loss: 0.08488 | train_chunk_0_accuracy: 0.9042  |  0:10:18s
epoch 8  | loss: 0.06252 | train_chunk_0_accuracy: 0.9044  |  0:11:30s
epoch 9  | loss: 0.04734 | train_chunk_0_accuracy: 0.9044  |  0:12:43s
epoch 10 | loss: 0.04345 | train_chunk_0_accuracy: 0.9073  |  0:14:00s
epoch 11 | loss: 0.03982 | train_chunk_0_accuracy: 0.90655 |  0:15:16s
epoch 12 | loss: 0.0395  | train_chunk_0_accuracy: 0.9197  |  0:16:32s
epoch 13 | loss: 0.03511 | train_chunk_0_accuracy: 0.99105 |  0:17:49s
epoch 

/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.36651 | train_chunk_1_accuracy: 0.907   |  0:01:11s
epoch 1  | loss: 0.20419 | train_chunk_1_accuracy: 0.907   |  0:02:22s
epoch 2  | loss: 0.17114 | train_chunk_1_accuracy: 0.907   |  0:03:39s
epoch 3  | loss: 0.15978 | train_chunk_1_accuracy: 0.907   |  0:04:56s
epoch 4  | loss: 0.13364 | train_chunk_1_accuracy: 0.907   |  0:06:12s
epoch 5  | loss: 0.09596 | train_chunk_1_accuracy: 0.98215 |  0:07:28s
epoch 6  | loss: 0.08382 | train_chunk_1_accuracy: 0.907   |  0:08:42s
epoch 7  | loss: 0.07194 | train_chunk_1_accuracy: 0.907   |  0:09:57s
epoch 8  | loss: 0.06179 | train_chunk_1_accuracy: 0.907   |  0:11:11s
epoch 9  | loss: 0.06014 | train_chunk_1_accuracy: 0.907   |  0:12:24s
epoch 10 | loss: 0.05779 | train_chunk_1_accuracy: 0.9107  |  0:13:34s
epoch 11 | loss: 0.04977 | train_chunk_1_accuracy: 0.91165 |  0:14:47s
epoch 12 | loss: 0.04491 | train_chunk_1_accuracy: 0.93225 |  0:16:03s
epoch 13 | loss: 0.03987 | train_chunk_1_accuracy: 0.91565 |  0:17:20s
epoch 

/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


...

/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.44763 | train_chunk_3_accuracy: 0.90346 |  0:00:43s
epoch 1  | loss: 0.24009 | train_chunk_3_accuracy: 0.90346 |  0:01:27s
epoch 2  | loss: 0.18823 | train_chunk_3_accuracy: 0.90346 |  0:02:07s
epoch 3  | loss: 0.16704 | train_chunk_3_accuracy: 0.90346 |  0:02:48s
epoch 4  | loss: 0.15445 | train_chunk_3_accuracy: 0.90346 |  0:03:29s
epoch 5  | loss: 0.13712 | train_chunk_3_accuracy: 0.90346 |  0:04:12s
epoch 6  | loss: 0.11516 | train_chunk_3_accuracy: 0.90346 |  0:04:56s
epoch 7  | loss: 0.09306 | train_chunk_3_accuracy: 0.90346 |  0:05:40s
epoch 8  | loss: 0.0699  | train_chunk_3_accuracy: 0.90346 |  0:06:25s
epoch 9  | loss: 0.05723 | train_chunk_3_accuracy: 0.90346 |  0:07:09s
epoch 10 | loss: 0.06219 | train_chunk_3_accuracy: 0.90346 |  0:07:51s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_train_chunk_3_accuracy = 0.90346


/home/server/anaconda3/envs/df/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Successfully saved model at tabnet_model.zip


'tabnet_model.zip'

**Data Split 70% and 15%, 15% for Training ,validation and testing**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Function to split the data and save as CSV
def split_and_save_data(file_path, target_column, test_size=0.3, random_state=42):
    # Read the full dataset
    df = pd.read_csv(file_path)

    # Define features and target variable
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Split the data into training and testing sets (randomized, stratified split to maintain class balance)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

    # Combine features and target variable back together for saving
    train_data = pd.concat([X_train, y_train], axis=1)
    test_data = pd.concat([X_test, y_test], axis=1)

    # Save train and test data as separate CSV files
    train_data.to_csv('train_data.csv', index=False)
    test_data.to_csv('test_data.csv', index=False)

    print("Train and test data saved as 'train_data.csv' and 'test_data.csv'.")

# Define file path and target column
file_path = 'dataset.csv'  # Replace with your actual CSV file path
target_column = 'vt_detection'  # Replace with your actual target column name

# Call the function to split and save the data
split_and_save_data(file_path, target_column)


Train and test data saved as 'train_data.csv' and 'test_data.csv'.


**Benin and Malware In full dataset target Column**

In [ ]:
df['vt_detection'].value_counts()

vt_detection
0    92134
1     9800
Name: count, dtype: int64

**Dataset Target Condition to create a target column**

In [ ]:
import pandas as pd

# Sample DataFrame creation

dfs=df
# Replace values: 1 if greater than or equal to 4, otherwise 0
dfs.loc[df['vt_detection'] < 4, 'vt_detection'] = 0 # Set all to 0
dfs.loc[df['vt_detection'] >= 4, 'vt_detection'] = 1  # Set to 1 where condition is met
print(df)

                                                   SHA256  \
0       080da3f89e42250d7462e17b40535cfca9b1a6a8370a31...   
1       461760796dd7789673cfaf68383da103033d54eb4a5267...   
2       dab8b14c3178b15200b23e47cecb9cc26b51c87d599ac0...   
3       db802025f9ec474d79793ac2aac556d2b52162ebc493e2...   
4       a44920abdd4915117412ad7695b8d95a1da5edfa513b09...   
...                                                   ...   
101929  67c2668b321c1cd06e3b4f54261e9bc24d0be1ece92ed5...   
101930  67c26a0ba216d216f652d6534225bc0534c4583fde6500...   
101931  67c272489d9a82dbdc73a2da78167e035d88764c98ba3f...   
101932  67c2744111f1ecdf28faf89ace9bf91d10163001db7bd7...   
101933  67c28aa0bb00161700683ca10cf25c2d1c50838bfd50fc...   

                       NOME                             PACOTE  API_MIN  API  \
0            2019 شاب دوزي‎          com.arabprod.aghani.douzi       10   26   
1                     Ishas  appinventor.ai_shameertanur.Ishas        7   28   
2                 Lashes&Go

In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=f65496d4b47762754e057146b7f3563935aeffdd8e5cc428c64729b3e00946e4
  Stored in directory: /root/.cache/pip/wheels/85/fa/a3/9c2d44c9f3cd77cf4e533b58900b2bf4487f2a17e8ec212a3d
Successfully built lime


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt

# Example dataset (Drebin Dataset)
data=pd.read_csv('/content/drive/MyDrive/MyResearchWork/drebin-215-dataset-5560malware-9476-benign_updated.csv')

X = data.drop('class', axis=1)  #replace with vt_detection for MH  dataset
y = data['class']

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



# Choose an instance to explain
instance_index = 0  # Index of the test sample to explain
instance = X_test.iloc[instance_index].values.reshape(1, -1)


# Initialize LIME explainer
explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns,
    class_names=['class'],
    mode="classification"
)
'''
# Train a model (replace with your model if needed)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Explain the prediction
explanation = explainer.explain_instance(
    data_row=instance.flatten(),
    predict_fn=model.predict_proba
)

# Visualize the explanation as a bar chart
explanation.as_pyplot_figure()
plt.title(f"Feature Contributions for Instance {instance_index}")
plt.tight_layout()
plt.show()

# Optional: Convert explanation to a heatmap-like visualization
import seaborn as sns

# Get feature contributions
explanation_dict = dict(explanation.as_list())
feature_contributions = pd.DataFrame({
    'Feature': explanation_dict.keys(),
    'Contribution': explanation_dict.values()
}).sort_values(by='Contribution', ascending=False)

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.barplot(data=feature_contributions, x='Contribution', y='Feature', palette="coolwarm")
plt.title(f"Feature Contributions for Instance {instance_index} (Heatmap Style)")
plt.xlabel("Contribution to Prediction")
plt.ylabel("Features")
plt.tight_layout()
plt.show()
'''

<ipython-input-3-688442efb679>:10: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv('/content/drive/MyDrive/MyResearchWork/drebin-215-dataset-5560malware-9476-benign_updated.csv')


TypeError: '<' not supported between instances of 'int' and 'str'

In [ ]:
########## For testing... Just to check what kind of results it get after applying LIME
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt

# Example dataset (replace this with your dataset)
from sklearn.datasets import load_iris
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a model (replace with your model if needed)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Choose an instance to explain
instance_index = 0  # Index of the test sample to explain
instance = X_test.iloc[instance_index].values.reshape(1, -1)

# Initialize LIME explainer
explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns,
    class_names=data.target_names,
    mode="classification"
)

# Explain the prediction
explanation = explainer.explain_instance(
    data_row=instance.flatten(),
    predict_fn=model.predict_proba
)

# Visualize the explanation as a bar chart
explanation.as_pyplot_figure()
plt.title(f"Feature Contributions for Instance {instance_index}")
plt.tight_layout()
plt.show()

# Optional: Convert explanation to a heatmap-like visualization
import seaborn as sns

# Get feature contributions
explanation_dict = dict(explanation.as_list())
feature_contributions = pd.DataFrame({
    'Feature': explanation_dict.keys(),
    'Contribution': explanation_dict.values()
}).sort_values(by='Contribution', ascending=False)

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.barplot(data=feature_contributions, x='Contribution', y='Feature', palette="coolwarm")
plt.title(f"Feature Contributions for Instance {instance_index} (Heatmap Style)")
plt.xlabel("Contribution to Prediction")
plt.ylabel("Features")
plt.tight_layout()
plt.show()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
######################################################################
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import lime
import lime.lime_tabular


data=pd.read_csv('/content/drive/MyDrive/MyResearchWork/data.csv')

X = data.drop('Result', axis=1)  #replace with vt_detection for MH  dataset
y = data['Result']

X=data.iloc[:, :-1]
y=data.iloc[:, -1]
# Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Initialize LIME explainer
explainer = lime.lime_tabular.LimeTabularExplainer(X_train.values,
                                                   feature_names=X.columns,
                                                   class_names=0,
                                                   discretize_continuous=True)

# Select a sample instance for explanation
sample_idx = 5  # Change index for different instances
sample_instance = X_test.iloc[sample_idx].values.reshape(1, -1)

flattened_instance = sample_instance.flatten()

print(type(sample_instance))
print(sample_instance.shape)
print(type(flattened_instance))
print(flattened_instance.shape)
# Get the prediction explanation
exp = explainer.explain_instance(sample_instance.flatten(), model.predict_proba, num_features=len(X.columns))

# Extract feature contributions
feature_names = []
feature_values = []
colors = []
'''
for feature, weight in exp.as_list():
    feature_names.append(feature)
    feature_values.append(weight)
    colors.append('green' if weight > 0 else 'red')  # Positive impact = green, Negative impact = red

# Plot feature contributions as a bar chart
plt.figure(figsize=(8, 5))
plt.barh(feature_names, feature_values, color=colors)
plt.xlabel("Feature Contribution")
plt.ylabel("Features")
plt.title(f"LIME Explanation for Prediction (Class: {iris.target_names[model.predict(sample_instance)[0]]})")
plt.axvline(0, color='black', linewidth=1)  # Reference line for zero contribution
plt.show()
'''

<class 'numpy.ndarray'>
(1, 86)
<class 'numpy.ndarray'>
(86,)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


TypeError: 'int' object is not iterable

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dct = DecisionTreeClassifier()

dct.fit(X_train, y_train)

Y_pred = dct.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix
print("Accuracy: ", accuracy_score(y_test, Y_pred))
print("Confusion_Matrix: \n", confusion_matrix(y_test, Y_pred))

from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier()

rfc.fit(X_train, y_train)

Y_pred = rfc.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix
print("Accuracy: ", accuracy_score(y_test, Y_pred))
print("Confusion_Matrix: \n", confusion_matrix(y_test, Y_pred))


Accuracy:  0.990909090909091
Confusion_Matrix: 
 [[23  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 18  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 14  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 16  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0 23  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 21  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 22  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 22  0  0  0  0  0  0  0  0  0  0  0  2  0]
 [ 0  0  0  0  0  0  0  0  0 22  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 19  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 23  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 22  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0 19  0  0  0  

In [ ]:
!pip install pytorch_tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

FOR CHECKING Whether it is CUDA Enabled or not:

In [ ]:
import torch
print(torch.cuda.is_available()
print(torch.cuda.get_device_name(0))
print(torch.cuda.device_count())


For LightGBM

In [ ]:
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier


# ==========================================================
# Function to Train and Evaluate LightGBM
# ==========================================================

def train_and_evaluate_lightgbm(
        train_file,
        valid_file,
        test_file,
        target_column):
    # ------------------------------------------------------
    # Read Training and Testing Data
    # ------------------------------------------------------
    train_df = pd.read_csv(train_file)
    valid_df = pd.read_csv(valid_file)
    test_df = pd.read_csv(test_file)

    # ------------------------------------------------------
    # Separate Features and Target
    # ------------------------------------------------------
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    X_valid = test_df.drop(columns=[target_column])
    y_valid = test_df[target_column]

    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    # ------------------------------------------------------
    # Encode Categorical Features
    # ------------------------------------------------------
    for column in X_train.select_dtypes(include=['object']).columns:
        encoder = LabelEncoder()
        X_train[column] = encoder.fit_transform(X_train[column])
        X_test[column] = X_test[column].apply(
            lambda x: encoder.transform([x])[0]
            if x in encoder.classes_
            else -1
        )

    # ------------------------------------------------------
    # Initialize LightGBM
    # ------------------------------------------------------

    lightgbm_model = LGBMClassifier(n_estimators=500,learning_rate=0.05,max_depth=-1,num_leaves=31,subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        objective='binary',
        boosting_type='gbdt',
        random_state=42,
        n_jobs=-1)

    # ------------------------------------------------------
    # Train Model
    # ------------------------------------------------------
    lightgbm_model.fit(X_train,y_train)
    test_predictions = lightgbm_model.predict(X_test)
    accuracy = accuracy_score(y_test,test_predictions)

    print("=" * 60)
    print("LightGBM Performance")
    print("=" * 60)
    print(f"Test Accuracy : {accuracy*100:.2f}%")

    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': lightgbm_model.feature_importances_
    })

    feature_importance = feature_importance.sort_values(
        by='Importance',
        ascending=False
    )

    print("\nTop 20 Important Features")
    print(feature_importance.head(20))

    # ------------------------------------------------------
    # Save Model
    # ------------------------------------------------------

    joblib.dump(lightgbm_model,"lightgbm_model.pkl")

    print("\nModel Saved Successfully.")
    return lightgbm_model


# ==========================================================
# Main Function
# ==========================================================

if __name__ == "__main__":
    train_file = "train_data.csv"
    valid_file = "valid_data.csv"
    test_file = "test_data.csv"
    target_column = "vt_detection"
    model = train_and_evaluate_lightgbm(
        train_file,
        valid_file,
        test_file,
        target_column
    )

Logistic Regression

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib

# Step 1: Train Logistic Regression on training data and save the model
def train_logistic_regression(train_file, target_column, model_file):
    # Load the training data
    train_df = pd.read_csv(train_file)

    # Separate features and target
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    # Encode categorical variables if necessary
    for column in X_train.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X_train[column] = le.fit_transform(X_train[column].astype(str))

    # Initialize and train the Logistic Regression model
    lr_model = LogisticRegression(max_iter=1000)  # You can adjust max_iter if needed
    lr_model.fit(X_train, y_train)

    # Save the trained model to a file
    joblib.dump(lr_model, model_file)
    print(f"Model saved to {model_file}")

    # Free memory by deleting the variables
    del train_df, X_train, y_train, lr_model

# Step 2: Load the saved model and evaluate on test data
def test_logistic_regression(test_file, target_column, model_file):
    # Load the test data
    test_df = pd.read_csv(test_file)

    # Separate features and target
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    # Encode categorical variables if necessary
    for column in X_test.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X_test[column] = le.fit_transform(X_test[column].astype(str))

    # Load the saved model
    lr_model = joblib.load(model_file)
    print("Model loaded successfully.")

    # Make predictions on the test data
    test_preds = lr_model.predict(X_test)

    # Calculate and display accuracy
    test_accuracy = accuracy_score(y_test, test_preds)
    print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

    return test_accuracy

# Define file paths and target column
train_file = 'train_data.csv'  # Path to your training data
valid_file = 'valid_data.csv'  # Path to your validation data
test_file = 'test_data.csv'  # Path to your test data
target_column = 'vt_detection'  # Replace with the name of your target column
model_file = 'logistic_regression_model.pkl'  # File to save the model

# Train the model
train_logistic_regression(train_file, target_column, model_file)

# Test the model on test data
test_accuracy = test_logistic_regression(test_file, target_column, model_file)

In [1]:
import sklearn, lightgbm, torch, numpy, pandas, matplotlib
print(sklearn.__version__)
print(lightgbm.__version__)
print(torch.__version__)
print(numpy.__version__)
print(pandas.__version__)
print(matplotlib.__version__)

1.6.1
4.6.0
2.11.0+cu128
2.0.2
2.2.2
3.10.0


Concatenation of prediction probabilities

In [ ]:
# TabNet probability predictions
P_tabnet_train = tabnet_model.predict_proba(X_train.values)

# LightGBM probability predictions
P_lgbm_train = lightgbm_model.predict_proba(X_train)

P_tabnet_test = tabnet_model.predict_proba(X_test.values)

P_lgbm_test = lightgbm_model.predict_proba(X_test)

In [ ]:
import numpy as np

# =====================================================
# Concatenate Probabilities (Training)
# =====================================================
P_combined_train = np.concatenate(
    (P_tabnet_train, P_lgbm_train),
    axis=1)

print("Combined Training Probability Shape:",
      P_combined_train.shape)

# =====================================================
# Concatenate Probabilities (Testing)
# =====================================================

P_combined_test = np.concatenate(
    (P_tabnet_test, P_lgbm_test),
    axis=1)

print("Combined Testing Probability Shape:",
      P_combined_test.shape)

Training the Logistic Regression meta-classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# =====================================================
# Meta Classifier
# =====================================================
meta_model = LogisticRegression(random_state=42,max_iter=1000)

meta_model.fit(P_combined_train,y_train)

final_predictions = meta_model.predict(P_combined_test)


accuracy = accuracy_score(y_test,final_predictions)

print(f"Meta Classifier Accuracy : {accuracy*100:.2f}%")

print(f"Stacking Accuracy : {accuracy*100:.2f}%")